In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 26.3 MB/s eta 0:00:00


In [2]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print("GPU Memory:", round(gpu.total_memory / 1024**3, 2), "GB")
else:
    print("No GPU available")

No GPU available


In [3]:
from datasets import Dataset

data = [
    {
        "question": "What products does Campus Essentials sell?",
        "answer": "Campus Essentials sells notebooks, pens, textbooks, USB drives, water bottles, and university merchandise."
    },
    {
        "question": "How much does a notebook cost?",
        "answer": "A standard notebook costs LKR 350."
    },
    {
        "question": "Do you sell computer science textbooks?",
        "answer": "Yes. Selected computer science textbooks are available, subject to stock availability."
    },
    {
        "question": "What time does the shop open?",
        "answer": "The shop opens at 8:30 AM from Monday to Friday."
    },
    {
        "question": "Can I return a damaged product?",
        "answer": "Yes. Customers can request an exchange for a damaged product by presenting the receipt within seven days of purchase."
    },
    {
        "question": "Do you sell USB drives?",
        "answer": "Yes. USB drives are available in selected storage capacities."
    },
    {
        "question": "Can I pay by card?",
        "answer": "Yes. The shop accepts cash and debit or credit cards."
    },
    {
        "question": "Are university-branded T-shirts available?",
        "answer": "Yes. University-branded T-shirts are available in selected sizes."
    },
    {
        "question": "Can I reserve a textbook?",
        "answer": "Yes. Customers can request a textbook reservation by contacting the shop. Reservations depend on stock availability."
    },
    {
        "question": "Where is Campus Essentials located?",
        "answer": "Campus Essentials is located beside the university library."
    }
]

print("Number of examples:", len(data))
dataset = Dataset.from_list(data)

Number of examples: 10


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )
else:
    dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map="auto"
)

print("Model loaded successfully!")
print("Parameters:", f"{model.num_parameters():,}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Parameters: 494,032,768


In [5]:
def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }

train_dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

print(train_dataset[0]["text"])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What products does Campus Essentials sell?<|im_end|>
<|im_start|>assistant
Campus Essentials sells notebooks, pens, textbooks, USB drives, water bottles, and university merchandise.<|im_end|>



In [6]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

print("LoRA configuration created!")

LoRA configuration created!


In [7]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [8]:
import torch
from trl import SFTTrainer, SFTConfig

# Use FP16 on a CUDA GPU to avoid the BF16 error
if torch.cuda.is_available():
    dtype = torch.float16
    print("GPU:", torch.cuda.get_device_name(0))
    print("Using FP16")
else:
    dtype = torch.float32
    print("No GPU available; using FP32")

training_args = SFTConfig(
    output_dir="./uni_fundamentals_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    bf16=False,
    gradient_checkpointing=True,
    max_length=512,
    dataset_text_field="text",
    report_to="none",
    seed=42
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer
)

trainer.train()

No GPU available; using FP32


Adding EOS to train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,3.033651


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=6, training_loss=2.853191395600637, metrics={'train_runtime': 136.5423, 'train_samples_per_second': 0.22, 'train_steps_per_second': 0.044, 'total_flos': 3664998637056.0, 'train_loss': 2.853191395600637, 'entropy': 2.0939979553222656, 'mean_token_accuracy': 0.7075471878051758, 'num_tokens': 1602.0, 'epoch': 3.0})

In [9]:
import trl
print("TRL version:", trl.__version__)

TRL version: 1.16.0


In [10]:
output_dir = "./uni_fundamentals_lora/final_adapter"

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("LoRA adapter saved to:", output_dir)

LoRA adapter saved to: ./uni_fundamentals_lora/final_adapter


In [11]:
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM

# These questions were NOT included in the 30 training examples.
test_questions = [
   "Do you sell laptop chargers?",
   "Can I buy stationery and textbooks from the same shop?",
]


def generate_answer(model, question):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode only the newly generated answer.
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

In [12]:
import gc

# Safely delete any existing models from memory if they exist
if "original_model" in globals():
    del original_model
if "model" in globals():
    del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map="auto"
)

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    output_dir
)

fine_tuned_answers = []

for question in test_questions:
    answer = generate_answer(fine_tuned_model, question)
    fine_tuned_answers.append(answer)

    print("\nQuestion:", question)
    print("Fine-tuned model:", answer)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Question: Do you sell laptop chargers?
Fine-tuned model: Yes, I can help you with your question.

Question: Can I buy stationery and textbooks from the same shop?
Fine-tuned model: Yes, you can buy stationery and textbooks from the same shop.


In [13]:
import pandas as pd

# Generate original model answers since they were not defined
original_answers = []
for question in test_questions:
    # Generate answers using the base model prior to applying adapters
    answer = generate_answer(base_model, question)
    original_answers.append(answer)

results = pd.DataFrame({
    "Question": test_questions,
    "Original Model": original_answers,
    "Fine-tuned Model": fine_tuned_answers
})

display(results)

,Question,Original Model,Fine-tuned Model
0,Do you sell laptop chargers?,"Yes, I can help you with your question.","Yes, I can help you with your question."
1,Can I buy stationery and textbooks from the sa...,"Yes, you can buy stationery and textbooks from...","Yes, you can buy stationery and textbooks from..."


In [14]:
!pip install -q sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 9.0 MB/s eta 0:00:00


In [15]:
test_questions = [
    "How much is the StudyPack Mini?",
    "What is the Wednesday notebook discount?",
    "How long are textbook reservations held?"
]

reference_answers = [
    "The StudyPack Mini costs LKR 875.",
    "Students receive a 13% discount on notebooks on Wednesdays.",
    "Textbook reservations are held for 36 hours."
]

In [16]:
original_answers = [
    generate_answer(base_model, q)
    for q in test_questions
]

finetuned_answers = [
    generate_answer(fine_tuned_model, q)
    for q in test_questions
]

In [17]:
import sacrebleu

original_bleu = sacrebleu.corpus_bleu(
    original_answers,
    [reference_answers]
)

finetuned_bleu = sacrebleu.corpus_bleu(
    finetuned_answers,
    [reference_answers]
)

print(f"Original BLEU:   {original_bleu.score:.2f}")
print(f"Fine-tuned BLEU: {finetuned_bleu.score:.2f}")

Original BLEU:   2.02
Fine-tuned BLEU: 2.02
